**Gaussian mixture models, latent variables, and EM**
-

**Libraries**
-

In [ ]:
from pathlib import Path
import sys

# Locate the project package when running from the project or notebooks folder.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.mixtures import GMMData, GMMSearch, GMMStability, GMMAnalysis
from src.models.gmm_plots import plot_search, plot_stability


## **1. Question of Interest and GMM Description**

**Question of Interest**
- There exists any probabilistic profiles in the orders from sales patterns and what is the stability of the groups when we use Gaussian Mixture Models? 



**GMM Description**

We are going to consider that all the orders data that we're going to use comes from a combination of K Gaussian distribution

$$
p(x_i)=\sum_{k=1}^{K}\pi_k\mathcal N(x_i\mid\mu_k,\Sigma_k)
$$

where 
$\pi_k$: is the weight of the component $k$, with $\sum_k\pi_k=1$.
$\mu_k$: is the mean vector of the component.
$\Sigma_k$: is the covariance matrix of the component.

In this sense, we don't know initially which orders belong to each component. That's why we suppose that there exist $k$ latent variables that will represent the profile of the orders, named $Z_i$. $Z_i$ represents a profile in the orders that we initially don't know but exist.

Therefore, we start doing expectations (E step) where we initialize or supposewe have some initial values for $\pi_k$,$\mu_k$ and $\Sigma_k$. Then, we calcule the probability that each observation belongs to each component

$$
\gamma_{ik}=P(Z_i=k\mid x_i)
$$

Finally, we apply Bayes Theorem to obtain:

$$
\boxed{
\gamma_{ik}=
\frac{\pi_k\mathcal N(x_i\mid\mu_k,\Sigma_k)}
{\sum_{j=1}^{K}\pi_j\mathcal N(x_i\mid\mu_j,\Sigma_j)}
}
$$

In this way, each order will have a probability of belonging to a certain component.

In consecuence, we continue with Maximization (step M) where we use the resposabilities of each the Estimation (step E) to update the parameters of each Gaussian Distribution. To to this, we calculate

$$
N_k=\sum_{i=1}^{n}\gamma_{ik}
$$

$N_k$ refers to the effective number of observations that are asigned to the k component. Then we update the weights, the mean vectors and covariances matrices of each Gaussian Distributions.

$$
\boxed{\pi_k=\frac{N_k}{n}}
$$

$$
\boxed{
\mu_k=\frac{\sum_{i=1}^{n}\gamma_{ik}x_i}{N_k}
}
$$
$$
\boxed{
\Sigma_k=
\frac{\sum_{i=1}^{n}\gamma_{ik}(x_i-\mu_k)(x_i-\mu_k)^T}{N_k}
}
$$

Finally, we alternate E and M steps to increase the log-likelihood until convergence.

## **2. Database Loading**

In [ ]:
data = GMMData()
data.frame.head()

In [ ]:
data.features

## **3. Hyperparameters Search**

### **3.1 Standarization**

In [ ]:
# Training-only scaling and the validation split are prepared by GMMData.
data.train.shape, data.validation.shape

### **3.2 Identifying an appropiate covariance and k_value**

In [ ]:
search = GMMSearch(n_jobs=-1).run(data)

### **3.3 Metrics Visualization**

In [ ]:
plot_search(search)

**Observations**
- Full and Diag covariance types are the ones with less balance with complexity and adjustment (AIC and BIC). These two have a better log-likelihood than spherical and tied, showiing a better adjustment in training and validation data.
- Choosing a GMM with \(k=4\) components is a defensible decision in terms of simplicity and interpretability. Although models with more components achieve better probabilistic fit according to AIC and BIC, they also increase model complexity and make the resulting clusters more difficult to interpret. On the other hand, $k=4$ presents a good balance in clustering metrics like Silhouette, Calinski-Harabas, Davies-Bouldin and Davies-Boulding, specially when the covariance_type is diag (excluding spherical and tied because of AIC and BIC).  
- The configuration that will be taken is covariance_type="diag" and $k=4$ because its simplicity in terms of interpretation and balance with adjustment and complexity.

### **3.4 Seed Estabilization**

In [ ]:
stability = GMMStability(k=4, covariance_type="diag", n_init_values=(1, 10)).run(data)
display(stability.runs.round(4))

In [ ]:
display(stability.summary.round(5))
plot_stability(stability)
stability.runs.loc[stability.runs["Train LogLik"].idxmax()]

**Observations**
- All models converged independently of the seed number.
- There is more stability between seeds when n_init=10 is implemented (ARI = 0.94) in constrast with n_init=1 (0.86)
- In n_init = 10, there was an adjustment decay in the seed 100 where the likelihood decreased more than two units. The partitions of seed 100 were very different in constrast with the other seeds that were taken.
- The use of multiple initializations helps to reduce the risk of congerge to less favorable local optima, but it doesn't guarantee finding the global optimum.

## **4. Clustering Analysis**

### **4.1 GMM**

In [ ]:
analysis = GMMAnalysis(data, k=4, covariance_type="diag", n_init=10).fit()

### **4.2 Convergence and EM Iterations**

In [ ]:
analysis.convergence_summary()

**Observations**

From the selected configuration $k=4$ and covariance_type="diag", we have:
- The algorithm converged in 12 iterations, which is lower than the maximum number of iterations allowed (300).
- The Log-likelihood in training and validation is very similar which indicates that the probabilistic adjustment is similar in both sets.

### **4.3 Estimation Step**

In [ ]:
analysis.responsibilities_summary().round(4)

**Observations**
- The mean of the probability is high which indicates that the observations in general have a strong probability of belonging to a certain component
- At mininum , 50% of the validation orders have a probability of belonging to a component with a very low or nule uncertainty.
- The 98% of the observations have a strong probability of belonging to a certain component (prob > 0.9).
- There exist a 0.3% of observations that have a max probability below 0.6 of belonging to a certain component.
- The mean normalized entropy is low which indicates that the uncertainty of the orders belonging to a certain component is low. 

### **4.4 Maximization Step**

In [ ]:
parameters = analysis.component_parameters()


**Observations**
- Component 0 has de 52% of the weight of the Gaussiang mixtures, followed by components 3 (29%), 1 (10%) and 2 (6%).
- Components 0 and 1 have item quantitys of 1 and 2 respectively, but they have almost 0 variance.
- The item profit ratio mean is negative por component 3 but positive in the other components, despite the variance is high for the feature in this component.
- The average and variance of the componentes in Order Discount Rate Feature are very similar.

### **4.5 Characterization**

#### **4.5.1 Cluster 0**

In [ ]:
analysis.analyze_cluster(0)


**Observations**
- The cluster contains the 53% of all the orders of validation.
- The sales range inside the cluster is high (min 9.9 max 1999.9), the standard deviation is around 154 USD.
- The cluster only contains orders with 1 item.
- The orders usually give profits.

In [ ]:
analysis.plot_cluster(0)


In [ ]:
analysis.category_counts(0, "Order Country", top=30)


**Observations**
- The predominant countries are EEUU, France, Mexico, Germany and Australia.

In [ ]:
analysis.category_counts(0, "Category Name")


**Observations**
- The predominant categories are Men's Footwear, Fishing, Water Sports and Camping & Hiking.

#### **4.5.2 Cluster 1**

In [ ]:
analysis.analyze_cluster(1)


**Observations**
- The cluster contains around de 10% of the validation orders.
- The sales that the company get with these orders are at maximum 200USD.
- The profitability is positive in at minimum the 75% of the data.
- The cluster only contains orders that have 2 items.
- The sales are usually around 100USD.

In [ ]:
analysis.plot_cluster(1)


In [ ]:
analysis.category_counts(1, "Order Country", top=30)


**Observations**
- The predominant countries are EEUU, Mexico, France, Germany and Brasil.

In [ ]:
analysis.category_counts(1, "Category Name")


**Observations**
- The predominant categories are Cleats, Women's Apparel, Indoor/Outdoor Games and Shop by Sport.

#### **4.5.3 Cluster 2**

In [ ]:
analysis.analyze_cluster(2)


**Observations**
- The cluster represents de 5.8% of the validation data.
- The sales are usually above 100 USD.
- The maximum at sales is 999.99 USD
- The profitability in all the orders inside the cluster is negative.
- The orders inside the cluster go from 1 to 5 items, but there is at least a 50% of the orders that have 3 or more products.

In [ ]:
analysis.plot_cluster(2)

In [ ]:
analysis.category_counts(2, "Order Country", top=30)

**Observations**
- The predominant countries are EEUU, France, Mexico and Germany.

In [ ]:
analysis.category_counts(2, "Category Name")


**Observations**
- The predominant categories are Cleats, Women's Apparel, ndoor/Outdoor Games, Cardio Equipment and Shop By Sport. 

#### **4.5.4 Cluster 3**

In [ ]:
analysis.analyze_cluster(3)


**Observations**
- The cluster 3 represents the 30% of the validation orders.
- The 75% of the sales are greater than 150 USD.
- The maximum at sales in the cluster is 500 USD.
- There is a 9.5% of negative profit observations in the cluster.
- The orders of the cluster have 3 or more items.

In [ ]:
analysis.plot_cluster(3)


In [ ]:
analysis.category_counts(3, "Order Country", top=30)


**Observations**
- The predominant countries are EEUU, Mexico, France, Alemania and Brasil.

In [ ]:
analysis.category_counts(3, "Category Name")


**Observations**
- The predominant categories are Cleats, Women's Apparel, Indoor/Outdoor Games, Cardio Equipment and Shop by Sport.

### **4.6 Identified Order Profiles**

**Profile 0 Mostly profitable single-item orders:** The largest group, representing about 53% of orders. Customers buy one item, usually generating a profit. Sales vary widely, including some expensive purchases.

**Profile 1 Small two-item orders:** About 10% of orders. Customers buy two items, typically spending around $100, with sales up to $200. Most orders are profitable.

**Profile 2 Loss-making orders:** About 5.8% of orders. Every order generates a loss, even though sales are often above $100. Quantities vary from one to five items.

**Profile 3 Mostly profitable orders with several items:** About 30% of orders. Customers buy three or more items, and most orders have sales above $150. Most are profitable, although approximately 9.5% generate losses.